# Introduction to Pipeline Processing System (PPS) Files
<hr style="border: 2px solid #f5bf03" />

- **Description:** This tutorial explains the basics of Pipeline Processing System (PPS) files for XMM-Newton.
- **Level:** Intermediate
- **Data:** XMM observation of SN 1006, an extended source (obsid=0653860101)
- **Requirements:** Must be run using pySAS version **2.3.0** or higher.
- **Credit:** Ryan Tanner (January 2026)
- **Support:** <a href="https://heasarc.gsfc.nasa.gov/docs/xmm/xmm_helpdesk.html">XMM Newton GOF Helpdesk</a>
- **Last verified to run:** 21 January 2026, for SAS v22.1 and pySAS v2.3.0

<hr style="border: 2px solid #f5bf03" />

<a id="introduction"></a>
## 1. Introduction

The raw data from XMM comes in `observation data files` (`ODFs`). Before you can begin analysis you first have to calibrate the ODF. To simplify things ESA provides `Pipeline Processing System` (`PPS`) files that provide event lists, images, spectra, and much, much more that are already calibrated and ready for analysis. PPS files are appropriate for many types of analysis (almost all types), but for very specialized analysis you may need to start over with the raw ODFs.

In this tutorial we will explain the types of products you can find in the PPS files.

<div class="alert alert-block alert-info">
    <b>Note:</b> This is only an <i>introduction</i> to PPS files. Complete information can be found in <a href="https://xmm-tools.cosmos.esa.int/external/xmm_obs_info/odf/data/docs/XMM-SOC-GEN-ICD-0024.pdf">the official Pipeline Products Description</a>. Users should consult that document for all information on the PPS files.
</div>

### Contents:

1. [Introduction](#introduction)
    1. [A Brief Note About PPS File Names](#noteaboutfilenames)
3. [Downloading PPS Files](#downloadingfiles)
    1. [Downloading All PPS Files](#downloadingallfiles)
    2. [Downloading a Subset of PPS Files](#downloadingsubset)
    3. [Get List of PPS Files](#getlistofppsfiles)
4. [The PPS Files](#ppsfiles)
    1. [Observation Summary File](#obssummary)
    2. [The Attitude and Calibration Index Files](#attandcalind)
    3. [The EPIC Event Lists](#epicevtlsts)
    4. [The EPIC FITS Image Files](#epicimages)
    5. [RGS Event Lists and Spectra](#rgsevtlstsspec)
5. [Discovering and Finding PPS File Types --  For Advanced Users](#advancedfilefinding)
    1. [How to Find Files Using Regular Expressions](#howtofind)

<a id="noteaboutfilenames"></a>
### 1.1 A Brief Note About PPS File Names

PPS files have the following convention for their names:

#### POOOOOOOOOODDZEEETTTTTTSXXX.FFF

(Now with spaces to show the different parts.)

#### P &nbsp;&nbsp; OOOOOOOOOO &nbsp;&nbsp; DD &nbsp;&nbsp; Z &nbsp;&nbsp; EEE &nbsp;&nbsp; TTTTTT &nbsp;&nbsp; S &nbsp;&nbsp; XXX &nbsp;&nbsp;.&nbsp;&nbsp; FFF

```
P          : All PPS Files start with a "P"
OOOOOOOOOO : 10 digit Obs ID
DD         : Instrument/Product Identifier
Z          : Exposure Flag
EEE        : Exposure Number
TTTTTT     : File/Product Type
S          : Data Subset Number/Character
XXX        : Source Number/Slew Step Number
FFF        : File Format
```

The Instrument/Product Identifier (`DD`) can take the values:

```
M1 : EPIC MOS 1 Camera
M2 : EPIC MOS 2 Camera
PN : EPIC PN Camera
EP : Combined EPIC Product
R1 : RGS 1 Camera
R2 : RGS 2 Camera
RG : Combined RGS Product
OM : Optical Monitor
CA : Catalogue Cross-Correlation File
OB : General Observation/Summary File
```

The Exposure Flag (`Z`) can take the values:

```
S : Scheduled Exposure
U : Unscheduled Exposure
X : File Not an Exposure (General File)
```

The File or Product Type (`TTTTTT`) can have a number of values. We will not list them all here (there are > 50!), but we will note a few important ones.

```
MIEVLI : EPIC MOS Event List
PIEVLI : EPIC PN Event List
IMAGE_ : EPIC Image File
EVENLI : RGS Event List
SRSPEC : RGS Spectra File
SUMMAR : Observation Summary File
ATTTSR : Spacecraft Attitude File
CALIND : Calibration Index File
```

The files can be in the following formats:

```
FIT : FITS File
FTZ : Gzipped FITS File
HTM : HTML File
PNG : PNG File
PDF : PDF File
ASZ : Gzipped ASCII File
ASC : ASCII File
```

If the files are gzipped, you **do not** have to unzip them. SAS will handle that.

#### Useful Links

- [`pysas` Documentation](https://xmm-tools.cosmos.esa.int/external/sas/current/doc/pysas/index.html "pysas Documentation")
- [`pysas` on GitHub](https://github.com/XMMGOF/pysas)
- [Common SAS Threads](https://www.cosmos.esa.int/web/xmm-newton/sas-threads/ "SAS Threads")
- [Users' Guide to the XMM-Newton Science Analysis System (SAS)](https://xmm-tools.cosmos.esa.int/external/xmm_user_support/documentation/sas_usg/USG/SASUSG.html "Users' Guide")
- [The XMM-Newton ABC Guide](https://heasarc.gsfc.nasa.gov/docs/xmm/abc/ "ABC Guide")
- [XMM Newton GOF Helpdesk](https://heasarc.gsfc.nasa.gov/docs/xmm/xmm_helpdesk.html "Helpdesk") - Link to form to contact the GOF Helpdesk.

<div class="alert alert-block alert-warning">
    <b>Warning:</b> By default this notebook will place observation data files in your default <tt>data_dir</tt> directory. Make sure pySAS has been configured properly.
</div>

<a id="downloadingfiles"></a>
## 2. Downloading PPS Files

In [ ]:
# pySAS imports
import pysas

# Useful imports
import os, re, glob
from IPython.display import HTML, Image, display
import xspec

# Astropy imports
from astropy.io import fits
from astropy.visualization import astropy_mpl_style

# Imports for plotting
import matplotlib.pyplot as plt
plt.style.use(astropy_mpl_style)

# To handle certain warnings
import warnings
warnings.filterwarnings("ignore")

To begin, you need the `Obs ID` you are working with. You use this Obs ID to initialize an instance of the `ObsID` class. We will call this instance of the `ObsID` class `my_pps`. `my_pps` contains the necessary functions to download the PPS files for our Obs ID.

In [ ]:
obsid = '0653860101'
my_pps = pysas.ObsID(obsid)

<a id="downloadingallfiles"></a>
### 2.1 Downloading All PPS Files

We download the data using the dedicated function `download_PPS_data`. This function uses `astroquery` to send a request for the specific Obs ID. This may take 1-5 minutes (or more!) depending on the Obs ID, your internet connection, if you are downloading to your local machine, or if you are using an online platform such as Fornax or DataLabs. By default `download_PPS_data` will not redownload previously downloaded PPS files unless the parameter `overwrite` is set to `True`.

In [ ]:
my_pps.download_PPS_data(repo='heasarc')

<a id="downloadingsubset"></a>
### 2.2 Downloading a Subset of PPS Files

The smallest number of PPS files a single Obs ID can have is 17 (which only happens if there is NO usable data for that Obs ID). But typically there are a few hundred to a few thousand (!) PPS files for each Obs ID. Depending of what you are doing you probably don't want to download *every* PPS file. Thus there is a way to download only a subset of the PPS files.

You can use the function `download_PPS_data` to download a subset of PPS files instead of ALL of the PPS files. A subset of files can be selected based on the identifiers in the file names (see Section 1 above) passed in as separate parameters. The parameters are:

```
instname     : Instrument/Product Identifier
expflag      : Exposure Flag
expno        : Exposure Number
product_type : File/Product Type
datasubsetno : Data Subset Number/Character
sourceno     : Source Number/Slew Step Number
extension    : File Format
filename     : Exact File Name
```

Here are a few examples of commands to download a subset of PPS files:

This will download all files for just the `MOS 1` camera.

```python
my_pps.download_PPS_data(instname='M1')
```

This will download all files with the `IMAGE_` type.

```python
my_pps.download_PPS_data(product_type='IMAGE_')
```

This will download all `PNG` files.

```python
my_pps.download_PPS_data(extension='PNG')
```

This will download all files for the `MOS 2`, from all unscheduled observations, of type `IMAGE_`, with a data subset number of `8` (`8` stands for full band products), with a file extension of `FTZ`.

```python
my_pps.download_PPS_data(instname='M2', expflag='U', product_type='IMAGE_', datasubsetno='8', extension='FTZ')
```

***

This will download a single file with the name `P0123700101OBX000SUMMAR0000.HTM`. This works if you know the full filename for a single file.

```python
my_pps.download_PPS_data(filename='P0123700101OBX000SUMMAR0000.HTM')
```

This method also accepts a list of exact filenames. Each file will be downloaded seperately.

```python
list_of_files = ['P0123700101OBX000SUMMAR0000.HTM', 'P0653860101PNS003PIEVLI0000.FTZ', 'P0653860101M1S001MIEVLI0000.FTZ', 'P0653860101R1S004EVENLI0000.FTZ']

my_pps.download_PPS_data(filename=list_of_files)
```

<div class="alert alert-block alert-info">
    <b>Note:</b> You can't use <b>both</b> the filename identifiers <i>and</i> the 'filename' parameter at the same time.
</div>

<a id="getlistofppsfiles"></a>
### 2.3 Get List of PPS Files

For advanced users there is the option of getting a list of all available PPS files for a given `Obs ID` using the function `get_all_PPS_filenames`. This will return a list of all available PPS files regardless of whether or not they have been downloaded. The list will be returned **and** stored as a variable in the `ObsID` object under the name `ALL_PPS_FILES`.

You can then filter this list and then download just the desired files.

In [ ]:
list_of_pps_files = my_pps.get_all_PPS_filenames()
print(f'Number of PPS Files: {len(list_of_pps_files)}')

<a id="ppsfiles"></a>
## 3. The PPS Files

After downloading the PPS files, pySAS will automatically parse the PPS data directory and store the filenames of several important files. These file names are stored as following variables in the `my_pps` object:
- `summary_file`
- `attitude_file`
- `calind_file`
- `EPIC_event_lists`
- `EPIC_images`
- `RGS_event_lists`
- `RGS_spectra`

<a id="obssummary"></a>
### 3.1 Observation Summary File

Let's take a look at the observation summary file.

In [ ]:
print(os.path.basename(my_pps.summary_file))

The leading '`P`' marks it as a PPS file, followed by the 10-digit `Obs ID`. The next two characters, '`OB`', indicate that it is a general observation file and not from a specific camera or detector. The next character, '`X`' is the exposure flag, but because this is not a file for a specific exposure it has the value '`X`'. The next three numbers are the exposure number, but becuase this is not a file for an exposure it has the value of '`000`'.

Then comes the file identifier, which is six characters long. In this case it is '`SUMMAR`' since this is the summary file for the Obs ID. The final four numbers are for data and source numbers and do not apply here, hence the value '`0000`'.

Finally there is the file extension, and this is an `html` file. The next cell will display the contents of the file (this will only display the contents, it will not render it as a normal html file, i.e. the internal links do not work). If you navigate to the location of the file you can open the file in a browser. This is the general observation summary file and contains general information about the observation. There are summary files for the PPS files, and for EPIC, RGS, and Optical Monitor data, along with cross correlations to major catalogs.

In [ ]:
with open(my_pps.summary_file, "r", encoding="utf-8") as f:
    html_content = f.read()

HTML(html_content)

<a id="attandcalind"></a>
### 3.2 The Attitude and Calibration Index Files

Two other important files are the spacecraft attitude and calibration index files. Various SAS tasks will need these files in order to perform their analysis.

In [ ]:
print(f'The spacecraft attitude file: {os.path.basename(my_pps.attitude_file)}')
print(f'The calibration index file  : {os.path.basename(my_pps.calind_file)}')

The structure of the file names is very similar to the observation summary file since these are all general observation files. The difference is that the attitude and calibration index files have the file designations of '`ATTTSR`' and '`CALIND`' respectively. They are also both FITS files. We will not display their contents here.

<a id="epicevtlsts"></a>
### 3.3 The EPIC Event Lists

In [ ]:
print('EPIC Event Lists: my_pps.EPIC_event_lists\n')
for file in my_pps.EPIC_event_lists:
    print(f' > {os.path.basename(file)}')

The two characters immediately after the Obs ID indicate which instrument these files come from. In this case `PN`, `M1`, and `M2` for the three EPIC cameras.

Below we display images generated from these event lists.

In [ ]:
for evtli in my_pps.EPIC_event_lists:
    my_pps.quick_eplot(evtli,vmin=1.0,vmax=100.0)

<a id="epicimages"></a>
### 3.4 The EPIC FITS Image Files

The `Pipeline Processing System` generates images (and other products) for different energy bands. The bands, and associated energy ranges, are found in the table below.

```
┌──╼╼╼╼╼╼╼╼╼╼╼┬╼╼╼╼╼╼╼╼╼╼╼╼╼╼╼╼╼╼╼┐
| Band Number | Energy Range (eV) |
├──╼╼╼╼╼╼╼╼╼╼╼┼╼╼╼╼╼╼╼╼╼╼╼╼╼╼╼╼╼╼╼┤
|      1      |    0.2 - 0.5      |
|      2      |    0.5 - 1.0      |
|      3      |    1.0 - 2.0      |
|      4      |    2.0 - 4.5      |
|      5      |    4.5 - 12.0     |
|      8      |    0.2 - 12.0     |
└──╼╼╼╼╼╼╼╼╼╼╼┴╼╼╼╼╼╼╼╼╼╼╼╼╼╼╼╼╼╼╼┘
```

The generated images come in two formats, FITS and PNG. We will be plotting the FITS image files. Again, in the file names the two characters immediately after the Obs ID indicate which camera they come from.

In [ ]:
print('\nEPIC FITS Image Files: my_pps.EPIC_images')
for file in my_pps.EPIC_images:
    print(f' > {file}')

Below we plot the images for the MOS 2 for all of the bands.

In [ ]:
bands = {'1' : '0.2 - 0.5',
         '2' : '0.5 - 1.0',
         '3' : '1.0 - 2.0',
         '4' : '2.0 - 4.5',
         '5' : '4.5 - 12.0',
         '8' : '0.2 - 12.0'}

for filename in my_pps.EPIC_images:
    if re.search('.*M2.*IMAGE_(\d).*',filename):
        band = re.findall('.*IMAGE_(\d).*',filename)[0]
        my_pps.quick_implot(filename, vmin=1.0, vmax=100.0, title=f'Energy Band {bands[band]} (keV)')

<a id="rgsevtlstsspec"></a>
### 3.5 RGS Event Lists and Spectra

The PPS files containg many files relevant to RGS analysis. At a basic level there are the event list and the source spectra files. 

In [ ]:
print('RGS Event Lists: my_pps.RGS_event_lists')
for file in my_pps.RGS_event_lists:
    print(f' > {file}')
print('\nRGS FITS Spectra: my_pps.RGS_spectra')
for file in my_pps.RGS_spectra:
    print(f' > {file}')

There are also whole field spectra files.

In [ ]:
whole_field_spec = []

for filename in my_pps.files['PPS']:
    if re.search('.*(R1|R2|RG).*WFSPEC.*FTZ',filename):
        print(filename)
        whole_field_spec.append(filename)

There are also response matrix files (`RMF`).

In [ ]:
rmfs = my_pps.return_file_list_on_pattern('.*WREMAT.*')

for file in rmfs:
    print(file)

Below using `PyXSPEC` we plot the whole field spectrum for the `RGS 1` instrument for the first order. First we have to sort through the files to get the sprectum and `rmf` file that we want. We can then load the spectrum and attatch the response file to the spectrum.

In [ ]:
for file in whole_field_spec:
    if re.search('.*R1.*1000.*',file):
        spec = file

print(f'Spectrum file: {spec}')

for file in rmfs:
    if re.search('.*R1.*1000.*',file):
        rmf = file

print(f'Reponse file : {rmf}')

xspec.AllData.clear()
s = xspec.Spectrum(spec)
s.response = rmf

def plot_spectrum(spectrum,plot_file_name='spectrum_plot.png'):
    xspec.Plot.device='/null'
    # xspec.Plot.xAxis = 'keV'
    xspec.Plot.xAxis = 'angstrom'

    # Pull off data for main plot
    xspec.Plot('data')
    energy = xspec.Plot.x()
    counts = xspec.Plot.y()
    xErrs = xspec.Plot.xErr()
    yErrs = xspec.Plot.yErr()

    # Make the figure and two subplots
    fig, ax0 = plt.subplots(figsize=(12, 6))

    # Main plot
    ax0.errorbar(energy, counts, yerr=yErrs, xerr=xErrs, linestyle='', marker='')
    # ax0.set_xscale('log')
    # ax0.set_yscale('log')
    ax0.tick_params(top=True,axis="x",direction="in",which='both')
    ax0.tick_params(axis="y",direction="in",which='both',right=True)
    ax0.set_ylabel('counts sec$^{-1}$ keV$^{-1}$')
    ax0.set_title('Data')

    # Save plot to file
    fig.savefig(plot_file_name)

In [ ]:
plot_spectrum(s)

<a id="advancedfilefinding"></a>
## 4. Discovering and Finding PPS File Types --  For Advanced Users

The <a href="https://xmm-tools.cosmos.esa.int/external/xmm_obs_info/odf/data/docs/XMM-SOC-GEN-ICD-0024.pdf">official Pipeline Products Description</a> document has tables showing the various PPS product types and descriptions of their contents. To aid in finding the various file types these tables have been converted into `.json` format and are stored as dictionaries in each `ObsID` instance created. The names of these dictionaries are:

- EPIC_PPS_products: File types for EPIC products.
- RGS_PPS_products: File types for RGS products.
- OBS_PPS_products: File types for general observation/summary products.
- OM_PPS_products: File types for Optical Monitor products.

<div class="alert alert-block alert-info">
    <b>Note:</b> The tables of file types for Cross Correlation Products have not been converted yet (as of pySAS 2.3.0)! They will be done in a future pySAS release!
</div>

As a reminder the filename convention for PPS files follows the format given below:

#### POOOOOOOOOODDZEEETTTTTTSXXX.FFF

The items for each product dictionary are themselves dictionaries containing exactly four items:

- Format (FFF): File format, FITS, PNG, HTML, etc.
- Product (TTTTTT): The PPS product indentifier.
- Source (DD): The product source, EPIC, RGS, General, etc.
- Description: A brief description of the product.

We will give a few examples to explain this.

The next cell will display the dictionary for EPIC products with the product type `IMAGE_` for the file `FITS` file type.

In [ ]:
my_pps.EPIC_PPS_products['IMAGE__FIT']

This is not to be confused with EPIC products with the product type `IMAGE_` for the file `PNG` file type.
<div class="alert alert-block alert-info">
    <b>Note:</b> Both have the product identifier '<tt>IMAGE_</tt>' but there are products that are '<tt>FITS</tt>' files and a set of corresponding '<tt>PNG</tt>' files.
</div>

In [ ]:
my_pps.EPIC_PPS_products['IMAGE__PNG']

If the observation is a `SLEW` observation the products are for the `PN` **only** and not the `MOS` cameras.

In [ ]:
my_pps.EPIC_PPS_products['IMAGE__SLEW']

The next cell will display **all** of the dictionary keys for the EPIC products and their corresponding descriptions.

In [ ]:
for key in my_pps.EPIC_PPS_products.keys():
    print('{0}: {1}'.format(key,my_pps.EPIC_PPS_products[key]['Description']))

The next cell will display **all** of the dictionary keys for the RGS products and their corresponding descriptions.

In [ ]:
for key in my_pps.RGS_PPS_products.keys():
    print('{0}: {1}'.format(key,my_pps.RGS_PPS_products[key]['Description']))

The next cell will display **all** of the dictionary keys for the General Observation products and their corresponding descriptions.

In [ ]:
for key in my_pps.OBS_PPS_products.keys():
    print('{0}: {1}'.format(key,my_pps.OBS_PPS_products[key]['Description']))

The next cell will display **all** of the dictionary keys for the Optical Monitor products and their corresponding descriptions.

In [ ]:
for key in my_pps.OM_PPS_products.keys():
    print('{0}: {1}'.format(key,my_pps.OM_PPS_products[key]['Description']))

<a id="howtofind"></a>
### 4.1 How to Find Files Using Regular Expressions

The `PPFFiles` object has a function `return_file_list_on_pattern` which will return a list of all PPS files matching a <a href="https://docs.python.org/3/library/re.html">`regular expression`</a> (<a href="https://docs.python.org/3/library/re.html">link to official Python documentation on regular expressions</a>) passed into the function.

For example, if I wanted a list of all of the EPIC image files in FITS format I could create a <a href="https://docs.python.org/3/library/re.html">`regular expression`</a> pattern using the '`Source`', '`Product`', and '`Format`' for the '`IMAGE_FIT`' dictionary.

In [ ]:
source      = my_pps.EPIC_PPS_products['IMAGE__FIT']['Source']
product     = my_pps.EPIC_PPS_products['IMAGE__FIT']['Product']
fileformat  = my_pps.EPIC_PPS_products['IMAGE__FIT']['Format']

pattern = f'.*{source}.*{product}.*{fileformat}'

print(f'File pattern to search for: "{pattern}"')

Passing this pattern into the function `return_file_list_on_pattern` will return a list of all PPS files matching that pattern.

> A brief note on regular expressions: The operator '`.*`' will match 0 or more characters. '`(M1|M2|PN)`' will match `M1`, `M2`, or `PN`.

<div class="alert alert-block alert-info">
    <b>Note:</b> You can pass in your own <tt>regular expression</tt> and have it return a corresponding list of files.
</div>

Basically all the function `return_file_list_on_pattern` is doing is running the following '`for`' loop:

```python
files = []
for filename in self.files['PPS']:
    if re.search(pattern,filename):
        files.append(filename)
```

<div class="alert alert-block alert-info">
    <b>Note:</b> If there are no correponding files for that product type, '<tt>return_file_list_on_pattern</tt>' will return an empty list.
</div>

In [ ]:
list_of_files = my_pps.return_file_list_on_pattern(pattern)
for file in list_of_files:
    print(file)

The following will find all EPIC source spectra files. **NOTE: There should be `591` source spectra files for this particular Obs ID (0653860101).**

In [ ]:
source      = my_pps.EPIC_PPS_products['SRSPEC_FIT']['Source']
product     = my_pps.EPIC_PPS_products['SRSPEC_FIT']['Product']
fileformat  = my_pps.EPIC_PPS_products['SRSPEC_FIT']['Format']

pattern = f'.*{source}.*{product}.*{fileformat}'

print(f'File pattern to search for: "{pattern}"\n')

list_of_files = my_pps.return_file_list_on_pattern(pattern)
for file in list_of_files:
    print(file)

You can use regular expressions to filter the list further. For example, to filter the above list by just the `MOS 1` files you could use:

In [ ]:
mos1_files = []
for file in list_of_files:
    if re.search('.*M1.*',file):
        mos1_files.append(file)
mos1_files

Too many lines of code? Don't worry we got you covered. The `ObsID` object has a private function `return_PPS_filenames` to handle this. Just pass in the product type dictionary and it will return a list of files.

In [ ]:
list_of_files = my_pps.return_PPS_filenames(my_pps.EPIC_PPS_products['IMAGE__FIT'])
for file in list_of_files:
    print(file)